# 4장. 기준선 모델

학습하지 않는 예측 방법은 얼마나 맞는가.

**준비:** 서울 열린데이터광장의 「서울시 지하철 호선별 역별 시간대별 승하차 인원 정보」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 『데이터 분석 완전 정복』 1장에서 받은 파일 그대로다.

## 4.1 단순 기준선: 누적 평균, 지난달 값, 작년 같은 달 값

In [ ]:
import os, unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)

DATA = "seoul_subway_hourly.csv"
if not os.path.exists(DATA):
    for f in os.listdir("."):
        if "승하차" in unicodedata.normalize("NFC", f):
            os.rename(f, DATA)

raw = pd.read_csv(DATA, encoding="cp949")
on = [c for c in raw.columns if c.endswith("승차인원")]
r = raw.drop_duplicates().drop_duplicates(["사용월", "호선명", "지하철역"]).copy()
r["승차"] = r[on].sum(axis=1)
r["역"] = r["호선명"] + " " + r["지하철역"]

m = r.pivot(index="사용월", columns="역", values="승차")
m.index = pd.PeriodIndex(m.index.astype(str), freq="M")
p = m.loc["2018-01":].dropna(axis=1)
p = p.loc[:, p.min() > 10000]
print(raw.shape, m.shape, p.shape)
print(p.iloc[-3:, :3].astype(int).to_string())

In [ ]:
TEST = p.loc["2025-01":].index
BOARD, FC = {}, {}

def off(actual, forecast):
    """역마다 달마다 예측이 실제에서 몇 % 벗어났는지 구해 그 중앙값을 돌려준다."""
    ratio = np.asarray(forecast, dtype=float) / np.asarray(actual, dtype=float) - 1
    return round(float(np.nanmedian(np.abs(ratio)) * 100), 2)

def grade(name, forecast, months=TEST):
    """예측 표를 받아 평가 구간의 오차율을 출력하고 BOARD와 FC에 기록한다."""
    FC[name] = forecast.loc[months]
    BOARD[name] = off(p.loc[months], forecast.loc[months])
    print(name, BOARD[name])

print(len(TEST), TEST[0], TEST[-1], p.shape[1] * len(TEST))

In [ ]:
so_far = p.expanding().mean().shift(1)      # 지난달까지의 평균
last_month = p.shift(1)                     # 지난달 값
last_year = p.shift(12)                     # 작년 같은 달 값

grade("누적 평균", so_far)
grade("지난달 값", last_month)
grade("작년 같은 달 값", last_year)

In [ ]:
!pip -q install holidays==0.108 koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def month_ticks(ax, months, step=1):
    ax.set_xticks(np.arange(len(months))[::step])
    ax.set_xticklabels([f"{t.year % 100}.{t.month}" for t in months][::step], fontsize=7, rotation=45)

In [ ]:
# 그림 4-2
total = p.sum(axis=1) / 1e6
span = p.loc["2023-01":].index
x = np.arange(len(span))
first = list(span).index(TEST[0])
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.axvspan(first - 0.5, len(span) - 0.5, color=to_rgba(NAVY, 0.05))
ax.plot(x, total.loc[span], color=NAVY, lw=2.0, marker="o", ms=3, label="실제")
ax.plot(x[first:], total.shift(12).loc[TEST], color=BRICK, lw=1.3, ls="--", marker="s", ms=2.5, label="작년 같은 달 값")
ax.plot(x[first:], total.shift(1).loc[TEST], color=GRAY, lw=1.3, marker="^", ms=2.5, label="지난달 값")
lo, hi = total.loc[span].min(), total.loc[span].max()
ax.set_ylim(lo - 14, hi + 14)
ax.text(first - 0.2, hi + 9, "평가 구간 (2025년 1월 ~ 2026년 10월)", fontsize=8, color=NAVY)
ax.text(0, lo - 10.5, "533개 역의 승차 인원을 더한 값이다. 본문의 오차율은 역마다 따로 계산한다", fontsize=7.5, color=GRAY)
ax.set_ylabel("월 승차 인원(백만 명)", fontsize=8)
month_ticks(ax, span, 2)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=3)
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_2.png", dpi=300)
plt.show()

## 4.2 달력 정보를 넣은 기준선: 추세 보정과 쉬는 날 보정

In [ ]:
miss = (last_year.loc[TEST] / p.loc[TEST] - 1)
monthly_bias = miss.median(axis=1) * 100
print(pd.DataFrame({"오차율": (miss.abs().median(axis=1) * 100).round(1),
                    "치우침": monthly_bias.round(1)}).T.to_string())

In [ ]:
# 그림 4-3
x = np.arange(len(TEST))
big = monthly_bias.abs() >= 5
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(x[~big], monthly_bias[~big], width=0.7, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.8)
ax.bar(x[big], monthly_bias[big], width=0.7, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.8)
for xi, v in zip(x[big], monthly_bias[big]):
    ax.text(xi, v + (0.6 if v > 0 else -0.6), f"{v:+.1f}", ha="center", va="bottom" if v > 0 else "top", fontsize=7.5, color=BRICK)
ax.axhline(0, color=GRAY, lw=0.8)
ax.set_ylim(-14, 14)
ax.text(-0.4, 12.2, "0보다 위: 예측이 실제보다 높았다", fontsize=7.5, color=GRAY)
ax.text(-0.4, -13.2, "0보다 아래: 예측이 실제보다 낮았다", fontsize=7.5, color=GRAY)
ax.set_ylabel("치우침(%)", fontsize=8)
month_ticks(ax, TEST)
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_3.png", dpi=300)
plt.show()

In [ ]:
recent = p.rolling(3).sum().shift(1) / p.rolling(3).sum().shift(13)   # 최근 3개월 합 ÷ 1년 전 같은 3개월 합
print(recent.loc[["2025-01", "2026-01"]].median(axis=1).round(3).to_string())

grade("작년 같은 달 × 추세", last_year * recent)

In [ ]:
import holidays

kr = holidays.KR(years=range(2018, 2027))

def rest_days(month):
    """그 달의 토요일, 일요일, 공휴일, 5월 1일을 센다."""
    days = pd.date_range(month.start_time, month.end_time, freq="D")
    return sum(d.dayofweek >= 5 or d.date() in kr or (d.month, d.day) == (5, 1) for d in days)

rest = pd.Series([rest_days(x) for x in p.index], index=p.index)
work = p.index.days_in_month - rest
print(pd.DataFrame({"평일": work, "쉬는 날": rest}).loc[["2024-01", "2025-01", "2024-09", "2025-09", "2024-10", "2025-10"]].T.to_string())

In [ ]:
def by_calendar(w):
    """쉬는 날 하루를 평일 w일로 환산해 달의 길이를 구하고, 하루당 승차로 바꿔서 작년 같은 달 값과 추세를 계산한다."""
    length = work + w * rest
    per_day = p.div(length, axis=0)
    flow = per_day.rolling(3).mean().shift(1) / per_day.rolling(3).mean().shift(13)
    return (per_day.shift(12) * flow).mul(length, axis=0)

BEFORE = p.loc["2023-01":"2024-12"].index
curve = pd.Series({round(w, 1): off(p.loc[BEFORE], by_calendar(w).loc[BEFORE]) for w in np.arange(0, 1.01, 0.1)})
print(curve.to_frame("2023~24년 오차율").T.to_string())

In [ ]:
# 그림 4-5
best = curve.idxmin()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(curve.index, curve.values, color=NAVY, lw=1.8, marker="o", ms=4.5)
ax.scatter([best], [curve[best]], s=70, color=BRICK, zorder=3)
ax.annotate(f"w = {best}: {curve[best]:.2f}%", (best, curve[best]), (best, curve[best] - 0.55), ha="center", fontsize=8, color=BRICK)
ax.annotate(f"쉬는 날을 평일과 같게 계산: {curve[1.0]:.2f}%", (1.0, curve[1.0]), (0.62, curve[1.0] + 0.25), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.annotate(f"쉬는 날을 0일로 계산: {curve[0.0]:.2f}%", (0.0, curve[0.0]), (0.08, curve[0.0] + 0.25), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.set_ylim(curve.min() - 0.9, curve.max() + 0.7)
ax.set_xticks(curve.index)
ax.set_xlabel("쉬는 날 하루를 평일 며칠로 환산하는가 (w)", fontsize=8)
ax.set_ylabel("2023~24년의 오차율(%)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_5.png", dpi=300)
plt.show()

In [ ]:
grade("달력 기준선", by_calendar(0.5))

In [ ]:
# 그림 4-6
def monthly(forecast):
    return (forecast.loc[TEST] / p.loc[TEST] - 1).abs().median(axis=1) * 100

x = np.arange(len(TEST))
a, b = monthly(last_year), monthly(by_calendar(0.5))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(x - 0.2, a, width=0.4, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.7, label="작년 같은 달 값")
ax.bar(x + 0.2, b, width=0.4, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.7, label="달력 기준선")
for xi in np.flatnonzero(a.to_numpy() >= 8):
    ax.text(xi - 0.2, a.iloc[xi] + 0.25, f"{a.iloc[xi]:.1f}", ha="center", fontsize=7.5, color=BRICK)
ax.set_ylim(0, 12.5)
ax.set_ylabel("그 달 오차율의 중앙값(%)", fontsize=8)
month_ticks(ax, TEST)
ax.legend(frameon=False, fontsize=8, loc="upper center", ncol=2)
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_6.png", dpi=300)
plt.show()

## 4.3 선형 회귀와 기준선 비교: 상대 오차와 MASE

In [ ]:
L = np.log(p)
LEN = np.log(work + 0.5 * rest)

def table(months, use_calendar):
    """달마다 역마다 한 줄. 지난달, 작년 같은 달, 작년 그 전달의 로그 승차와, 달 길이의 전년 대비 변화."""
    X, y = [], []
    for t in months:
        i = p.index.get_loc(t)
        cols = [np.ones(p.shape[1]), L.iloc[i - 1], L.iloc[i - 12], L.iloc[i - 13]]
        if use_calendar:
            cols += [np.full(p.shape[1], LEN.iloc[i] - LEN.iloc[i - 12]),
                     np.full(p.shape[1], LEN.iloc[i - 1] - LEN.iloc[i - 13])]
        X.append(np.column_stack(cols))
        y.append(L.iloc[i].to_numpy())
    return np.vstack(X), np.concatenate(y)

LEARN = p.loc["2022-02":"2024-12"].index

def learned(name, use_calendar):
    Xtr, ytr = table(LEARN, use_calendar)
    Xte, yte = table(TEST, use_calendar)
    coef = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]
    FC[name] = pd.DataFrame(np.exp(Xte @ coef).reshape(len(TEST), -1), index=TEST, columns=p.columns)
    BOARD[name] = off(p.loc[TEST], FC[name])
    print(name, len(ytr), off(np.exp(ytr), np.exp(Xtr @ coef)), BOARD[name])
    print(coef.round(3).tolist())

learned("선형 회귀", False)
learned("선형 회귀 + 달력", True)

In [ ]:
board = pd.Series(BOARD, name="오차율").to_frame()
for base in ["지난달 값", "작년 같은 달 값", "달력 기준선"]:
    board[f"÷ {base}"] = (board["오차율"] / board.loc[base, "오차율"]).round(2)
print(board.to_string())

In [ ]:
# 그림 4-7
order = board["오차율"].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.9))
ypos = np.arange(len(order))
for yi, (name, val) in zip(ypos, order.items()):
    col = BRICK if name.startswith("선형 회귀") else NAVY
    ax.barh(yi, val, height=0.6, color=to_rgba(col, 0.55 if col == BRICK else 0.22), edgecolor=col, lw=0.9)
    ratio = board.loc[name, "÷ 작년 같은 달 값"]
    ax.text(val + 0.12, yi, f"{val:.2f}%   (작년 같은 달 값의 {ratio:.2f}배)", va="center", fontsize=8, color=col)
ax.set_yticks(ypos, order.index, fontsize=8)
ax.set_xlim(0, 13.5)
ax.set_xlabel("2025년 1월 ~ 2026년 10월의 오차율(%). 남색은 기준선, 벽돌색은 계수를 학습한 식", fontsize=8)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_4_7.png", dpi=300)
plt.show()

In [ ]:
scale = (p.loc[BEFORE] - last_year.loc[BEFORE]).abs().mean()   # 역마다: 2023~24년에 작년 같은 달 값으로 답했을 때의 평균 절대 오차

rows = []
for name, forecast in FC.items():
    mase = (p.loc[TEST] - forecast).abs().mean() / scale
    rows.append((name, round(float(mase.median()), 2), round(float((mase < 1).mean() * 100), 1)))
print(pd.DataFrame(rows, columns=["방법", "MASE의 중앙값", "MASE < 1인 역(%)"]).set_index("방법").to_string())

In [ ]:
def per_station(name):
    return (FC[name] / p.loc[TEST] - 1).abs().median() * 100      # 역마다 22개월 오차율의 중앙값

base, model = per_station("달력 기준선"), per_station("선형 회귀 + 달력")
print(len(base), int((base < model).sum()), round(float((base < model).mean() * 100), 1))
print(round(float(base.median()), 2), round(float(model.median()), 2), round(float((model - base).median()), 2))

## 4.4 기준선이 맞지 않는 시기: 2020년과 2021년

In [ ]:
flow_only = last_year * recent
rows = []
for year in ["2019", "2020", "2021", "2025"]:
    months = p.loc[year].index
    rows.append((year, off(p.loc[months], last_month.loc[months]), off(p.loc[months], last_year.loc[months]),
                 off(p.loc[months], flow_only.loc[months]), off(p.loc[months], by_calendar(0.5).loc[months])))
years = pd.DataFrame(rows, columns=["연도", "지난달 값", "작년 같은 달 값", "× 추세", "달력 기준선"]).set_index("연도")
print(years.to_string())

In [ ]:
months = p.loc["2020"].index
print(pd.DataFrame({"지난달 값": ((last_month.loc[months] / p.loc[months] - 1).abs().median(axis=1) * 100).round(1),
                    "작년 같은 달 값": ((last_year.loc[months] / p.loc[months] - 1).abs().median(axis=1) * 100).round(1)}).T.to_string())

In [ ]:
# 그림 4-8
span = p.loc["2019-01":"2021-12"].index
x = np.arange(len(span))
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(x, total.loc[span], color=NAVY, lw=2.0, marker="o", ms=3, label="실제")
ax.plot(x, total.shift(12).loc[span], color=BRICK, lw=1.3, ls="--", marker="s", ms=2.5, label="작년 같은 달 값")
ax.plot(x, total.shift(1).loc[span], color=GRAY, lw=1.3, marker="^", ms=2.5, label="지난달 값")
k = list(span).index(pd.Period("2020-03", freq="M"))
ax.annotate("2020년 3월\n작년 같은 달 값은 2019년 3월의 승차 인원이다", (k, total.shift(12).loc["2020-03"]), (k + 1.5, total.loc[span].max() + 16),
            fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate("지난달 값은 한 달 늦게 실제를 따라간다", (k + 1, total.shift(1).loc["2020-04"]), (k + 4, total.loc[span].min() - 12),
            fontsize=8, color=GRAY, arrowprops={"arrowstyle": "->", "color": GRAY, "lw": 0.8})
ax.set_ylim(total.loc[span].min() - 20, total.loc[span].max() + 28)
ax.set_ylabel("월 승차 인원(백만 명)", fontsize=8)
month_ticks(ax, span, 2)
ax.legend(frameon=False, fontsize=8, loc="lower left", ncol=3)
frame(ax)
fig.tight_layout()
fig.savefig("fig_4_8.png", dpi=300)
plt.show()